# DESeq2 Results Visualization

This notebook generates standard visualizations for differential expression results:
- Volcano plot
- MA plot
- Top genes heatmap
- PCA plot (if count matrix provided)

**Inputs**: DESeq2 results table (TSV), optionally normalized counts
**Outputs**: Publication-ready figures (PNG)

In [ ]:
# Parameters (can be overridden by papermill)
PVALUE_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0
TOP_GENES = 50
FIGURE_DPI = 150

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']} ({inp['size']} bytes)")

In [ ]:
# Load DESeq2 results
de_results = get(0)  # First input should be DE results
print(f"Loaded {len(de_results)} genes")
de_results.head()

In [ ]:
# Standardize column names (DESeq2 output varies)
column_mapping = {
    'log2FoldChange': 'log2FC',
    'log2foldchange': 'log2FC',
    'logFC': 'log2FC',
    'padj': 'padj',
    'adj.P.Val': 'padj',
    'FDR': 'padj',
    'pvalue': 'pvalue',
    'PValue': 'pvalue',
    'baseMean': 'baseMean',
    'AveExpr': 'baseMean'
}

de_results = de_results.rename(columns=column_mapping)

# Add gene column if not present
if 'gene' not in de_results.columns:
    if de_results.index.name:
        de_results['gene'] = de_results.index
    elif 'Gene' in de_results.columns:
        de_results['gene'] = de_results['Gene']
    elif 'gene_id' in de_results.columns:
        de_results['gene'] = de_results['gene_id']

## 1. Volcano Plot

In [ ]:
def create_volcano_plot(df, pval_thresh=0.05, lfc_thresh=1.0):
    """Create publication-ready volcano plot."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate -log10(padj)
    df = df.copy()
    df['neg_log10_padj'] = -np.log10(df['padj'].clip(lower=1e-300))
    
    # Classify genes
    df['category'] = 'Not significant'
    df.loc[(df['padj'] < pval_thresh) & (df['log2FC'] > lfc_thresh), 'category'] = 'Up'
    df.loc[(df['padj'] < pval_thresh) & (df['log2FC'] < -lfc_thresh), 'category'] = 'Down'
    
    # Color palette
    colors = {'Not significant': '#BBBBBB', 'Up': '#E74C3C', 'Down': '#3498DB'}
    
    # Plot each category
    for cat, color in colors.items():
        mask = df['category'] == cat
        ax.scatter(df.loc[mask, 'log2FC'], df.loc[mask, 'neg_log10_padj'],
                   c=color, label=f"{cat} ({mask.sum()})", alpha=0.6, s=20)
    
    # Add threshold lines
    ax.axhline(-np.log10(pval_thresh), color='gray', linestyle='--', linewidth=1)
    ax.axvline(lfc_thresh, color='gray', linestyle='--', linewidth=1)
    ax.axvline(-lfc_thresh, color='gray', linestyle='--', linewidth=1)
    
    # Labels
    ax.set_xlabel('log2 Fold Change', fontsize=12)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
    ax.set_title('Volcano Plot', fontsize=14)
    ax.legend(loc='upper right')
    
    # Label top genes
    top_up = df[(df['category'] == 'Up')].nlargest(5, 'neg_log10_padj')
    top_down = df[(df['category'] == 'Down')].nlargest(5, 'neg_log10_padj')
    
    for _, row in pd.concat([top_up, top_down]).iterrows():
        if 'gene' in row:
            ax.annotate(row['gene'], (row['log2FC'], row['neg_log10_padj']),
                       fontsize=8, alpha=0.8)
    
    plt.tight_layout()
    return fig

fig_volcano = create_volcano_plot(de_results, PVALUE_THRESHOLD, LOG2FC_THRESHOLD)
plt.show()

In [ ]:
# Save volcano plot
put(fig_volcano, 'volcano_plot.png')
print("Saved: volcano_plot.png")

## 2. MA Plot

In [ ]:
def create_ma_plot(df, pval_thresh=0.05):
    """Create MA plot (log2FC vs mean expression)."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    df = df.copy()
    df['significant'] = df['padj'] < pval_thresh
    
    # Plot
    ax.scatter(np.log10(df.loc[~df['significant'], 'baseMean'] + 1),
               df.loc[~df['significant'], 'log2FC'],
               c='#BBBBBB', alpha=0.5, s=10, label='Not significant')
    ax.scatter(np.log10(df.loc[df['significant'], 'baseMean'] + 1),
               df.loc[df['significant'], 'log2FC'],
               c='#E74C3C', alpha=0.6, s=15, label='Significant')
    
    ax.axhline(0, color='gray', linestyle='-', linewidth=1)
    ax.set_xlabel('log10(Mean Expression)', fontsize=12)
    ax.set_ylabel('log2 Fold Change', fontsize=12)
    ax.set_title('MA Plot', fontsize=14)
    ax.legend()
    
    plt.tight_layout()
    return fig

if 'baseMean' in de_results.columns:
    fig_ma = create_ma_plot(de_results, PVALUE_THRESHOLD)
    plt.show()
    put(fig_ma, 'ma_plot.png')
    print("Saved: ma_plot.png")
else:
    print("Skipping MA plot: baseMean column not found")

## 3. Top Genes Heatmap

In [ ]:
def create_top_genes_heatmap(df, n_genes=50):
    """Create heatmap of top differentially expressed genes."""
    # Get top genes by adjusted p-value
    top_genes = df.dropna(subset=['padj']).nsmallest(n_genes, 'padj')
    
    # Create a simple heatmap of log2FC
    fig, ax = plt.subplots(figsize=(8, max(10, n_genes * 0.25)))
    
    # Sort by log2FC for better visualization
    top_genes = top_genes.sort_values('log2FC')
    
    # Create heatmap data
    gene_names = top_genes['gene'].values if 'gene' in top_genes.columns else top_genes.index
    values = top_genes['log2FC'].values.reshape(-1, 1)
    
    # Plot
    cmap = sns.diverging_palette(240, 10, as_cmap=True)
    vmax = max(abs(values.min()), abs(values.max()))
    
    im = ax.imshow(values, cmap=cmap, aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_yticks(range(len(gene_names)))
    ax.set_yticklabels(gene_names, fontsize=8)
    ax.set_xticks([0])
    ax.set_xticklabels(['log2FC'])
    ax.set_title(f'Top {n_genes} Differentially Expressed Genes', fontsize=14)
    
    plt.colorbar(im, ax=ax, label='log2 Fold Change')
    plt.tight_layout()
    return fig

fig_heatmap = create_top_genes_heatmap(de_results, TOP_GENES)
plt.show()
put(fig_heatmap, 'top_genes_heatmap.png')
print("Saved: top_genes_heatmap.png")

## 4. Summary Statistics

In [ ]:
# Calculate summary statistics
total_genes = len(de_results)
sig_genes = (de_results['padj'] < PVALUE_THRESHOLD).sum()
up_genes = ((de_results['padj'] < PVALUE_THRESHOLD) & (de_results['log2FC'] > LOG2FC_THRESHOLD)).sum()
down_genes = ((de_results['padj'] < PVALUE_THRESHOLD) & (de_results['log2FC'] < -LOG2FC_THRESHOLD)).sum()

summary = f"""
# Differential Expression Summary

## Parameters
- P-value threshold: {PVALUE_THRESHOLD}
- log2FC threshold: {LOG2FC_THRESHOLD}

## Results
- Total genes tested: {total_genes:,}
- Significant genes (padj < {PVALUE_THRESHOLD}): {sig_genes:,} ({sig_genes/total_genes*100:.1f}%)
- Upregulated (log2FC > {LOG2FC_THRESHOLD}): {up_genes:,}
- Downregulated (log2FC < -{LOG2FC_THRESHOLD}): {down_genes:,}
"""

print(summary)
put(summary, 'de_summary.md')

In [ ]:
print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - volcano_plot.png")
print("  - ma_plot.png (if baseMean available)")
print("  - top_genes_heatmap.png")
print("  - de_summary.md")